#Set-up

In [9]:
!pip install transformer-lens

In [10]:
import torch
import numpy as np
from transformer_lens import HookedTransformer
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [11]:
print("Loading GPT-2 Small...")
model = HookedTransformer.from_pretrained("gpt2-small")
print(f"Model loaded!")
print(f"Number of layers: {model.cfg.n_layers}")
print(f"Number of heads per layer: {model.cfg.n_heads}")
print(f"Residual stream dimension: {model.cfg.d_model}")
print(f"Total attention heads: {model.cfg.n_layers * model.cfg.n_heads}")

Loading GPT-2 Small...
Loaded pretrained model gpt2-small into HookedTransformer
Model loaded!
Number of layers: 12
Number of heads per layer: 12
Residual stream dimension: 768
Total attention heads: 144


# Visualizing Attention

In [12]:
text = "When Mary and John went to the store, Mary gave John"
print(f"Input text: {text}")

# Convert to tokens and run through model
tokens = model.to_tokens(text)
print(f"\nTokens shape: {tokens.shape}")
print(f"Number of tokens: {tokens.shape[1]}")

# Run model and cache all activations
logits, cache = model.run_with_cache(tokens)

# Extract attention patterns
# Shape: [batch, num_heads, seq_len, seq_len]
attention_patterns = cache["pattern", 0]  # Layer 0 attention patterns
print(f"\nAttention patterns shape: {attention_patterns.shape}")
print(f"This means: [batch_size={attention_patterns.shape[0]}, num_heads={attention_patterns.shape[1]}, query_pos={attention_patterns.shape[2]}, key_pos={attention_patterns.shape[3]}]")

# Get token strings for visualization
str_tokens = model.to_str_tokens(text)
print(f"\nToken strings: {str_tokens}")

Input text: When Mary and John went to the store, Mary gave John

Tokens shape: torch.Size([1, 13])
Number of tokens: 13

Attention patterns shape: torch.Size([1, 12, 13, 13])
This means: [batch_size=1, num_heads=12, query_pos=13, key_pos=13]

Token strings: ['<|endoftext|>', 'When', ' Mary', ' and', ' John', ' went', ' to', ' the', ' store', ',', ' Mary', ' gave', ' John']


In [13]:
def plot_attention_pattern(attention_pattern, tokens, layer, head):
    """
    Visualize a single attention head's pattern
    attention_pattern: [seq_len, seq_len]
    """
    # Ensure tokens are strings and handle length
    token_labels = [f"{i}:{tok}" for i, tok in enumerate(tokens)]

    fig = px.imshow(
        attention_pattern,
        labels=dict(x="Key Position (TO)", y="Query Position (FROM)", color="Attention"),
        x=token_labels,
        y=token_labels,
        color_continuous_scale="Blues",
        title=f"Layer {layer}, Head {head} - Induction Pattern",
        aspect="auto"
    )
    fig.update_xaxes(side="top")
    fig.update_layout(height=600, width=700)
    return fig

# Visualize a few heads from layer 0
layer_idx = 0
attention_layer_0 = cache["pattern", layer_idx][0]  # [num_heads, seq_len, seq_len]

# Plot head 0, 3, and 7 from layer 0
for head_idx in [0, 3, 7]:
    attention_pattern = attention_layer_0[head_idx].cpu().numpy()
    fig = plot_attention_pattern(attention_pattern, str_tokens, layer_idx, head_idx)
    fig.show()

print("- Diagonal patterns mean 'attend to yourself'")
print("- Vertical lines mean 'one position is very important'")
print("- Triangular patterns mean 'attend to all previous tokens equally'")

- Diagonal patterns mean 'attend to yourself'
- Vertical lines mean 'one position is very important'
- Triangular patterns mean 'attend to all previous tokens equally'


#Induction heads

In [14]:
# sequence with repeated tokens
induction_text = "The cat sat on the mat. The cat"
print(f"Induction test sequence: {induction_text}")

# Run through model
induction_tokens = model.to_tokens(induction_text)
induction_logits, induction_cache = model.run_with_cache(induction_tokens)

# Get token strings
induction_str_tokens = model.to_str_tokens(induction_text)
print(f"Tokens: {induction_str_tokens}")

# Function to compute induction score for each head
def compute_induction_score(attention_pattern, repeated_token_pos):
    """
    Measure if a head attends from current position back to
    the token that came after the previous instance of current token

    For "The cat sat on the mat. The cat"
                                      ^
    At final "cat", does it attend to "sat" (which came after first "cat")?
    """
    # This is simplified - just check if pattern shows the induction behavior
    # At the repeated token position, check attention to the relevant past position

    # For demo: measure attention from last token to positions after first occurrence
    last_pos = attention_pattern.shape[0] - 1

    # Sum attention to positions 2-5 (after first "cat" which is at position 1)
    induction_score = attention_pattern[last_pos, 2:7].sum().item()

    return induction_score

# Compute induction scores for all heads in all layers
print("INDUCTION SCORES BY HEAD")

induction_scores = []
for layer in range(model.cfg.n_layers):
    layer_patterns = induction_cache["pattern", layer][0]  # [n_heads, seq, seq]

    for head in range(model.cfg.n_heads):
        pattern = layer_patterns[head]
        score = compute_induction_score(pattern, repeated_token_pos=1)
        induction_scores.append({
            'layer': layer,
            'head': head,
            'score': score
        })

# Sort by score and show top candidates
import pandas as pd
df_scores = pd.DataFrame(induction_scores)
df_scores = df_scores.sort_values('score', ascending=False)

print("\nTop 10 Induction Head Candidates:")
print(df_scores.head(10))

for idx in range(min(3, len(df_scores))):
    top = df_scores.iloc[idx]
    layer, head = int(top['layer']), int(top['head'])

    pattern = induction_cache["pattern", layer][0, head].cpu().numpy()
    fig = plot_attention_pattern(pattern, induction_str_tokens, layer, head)
    fig.update_layout(title=f"Layer {layer}, Head {head} - Induction Score: {top['score']:.3f}")
    fig.show()


Induction test sequence: The cat sat on the mat. The cat
Tokens: ['<|endoftext|>', 'The', ' cat', ' sat', ' on', ' the', ' mat', '.', ' The', ' cat']
INDUCTION SCORES BY HEAD

Top 10 Induction Head Candidates:
     layer  head     score
65       5     5  0.840103
68       5     8  0.794466
61       5     1  0.719756
36       3     0  0.715208
60       5     0  0.696668
142     11    10  0.645039
5        0     5  0.563056
23       1    11  0.560211
1        0     1  0.527747
143     11    11  0.518807


#Logit difference

**In GTP-2 small: **

Vocabulary size = 50,257 tokens

Logits = a vector of 50,257 numbers

**What are logits:** Logits are the raw and unnormalized scores that a neural network outputs before converting them to probabilities.
They are like "confidence scores" can be (positive, negative, large, small).

**Def:** Metric to measure performance on a task. Measures how much the model prefers the correct answer over a plausible incorrect answer. It's calculated as:

Logit Difference = logit(correct) - logit(incorrect)

When we ablate (disable) components:
- If logit diff drops → that component was important for the task
- If logit diff stays high → that component wasn't critical
- This proves causally which parts of the circuit matter

In [15]:


ioi_prompt = "When Mary and John went to the store, Mary gave"
print(f"Test prompt: {ioi_prompt}")

# Run through model
ioi_tokens = model.to_tokens(ioi_prompt)
ioi_logits, ioi_cache = model.run_with_cache(ioi_tokens)

# Get the logits at the final position (where we predict next token)
final_logits = ioi_logits[0, -1, :]
print(f"\nLogits shape: {final_logits.shape}") # vocab_size]
print(f"This is a score for each of the {final_logits.shape[0]} tokens in vocabulary")

# Find token IDs for "John" and "Mary"
john_token = model.to_tokens(" John", prepend_bos=False)[0, 0]
mary_token = model.to_tokens(" Mary", prepend_bos=False)[0, 0]

print(f"\nToken IDs:")
print(f"' John' token ID: {john_token}")
print(f"' Mary' token ID: {mary_token}")

# Get their logits
john_logit = final_logits[john_token].item()
mary_logit = final_logits[mary_token].item()

print(f"\nLogit values:")
print(f"Logit for ' John' (correct IO): {john_logit:.3f}")
print(f"Logit for ' Mary' (subject, incorrect): {mary_logit:.3f}")

# Calculate logit difference
logit_diff = john_logit - mary_logit

print(f"LOGIT DIFFERENCE: {logit_diff:.3f}")


if logit_diff > 0:
    print(f"Model prefers 'John' by {logit_diff:.3f} logits")
    print(f"  This means the model correctly favors the indirect object!")
else:
    print(f"Model prefers 'Mary' by {abs(logit_diff):.3f} logits")
    print(f"  The model is confused about the task")

# Convert to probabilities
import torch.nn.functional as F
probs = F.softmax(final_logits, dim=0)
john_prob = probs[john_token].item()
mary_prob = probs[mary_token].item()

print(f"\nAs probabilities:")
print(f"P('John') = {john_prob:.4f} = {john_prob*100:.2f}%")
print(f"P('Mary') = {mary_prob:.4f} = {mary_prob*100:.2f}%")

# Visualize top predictions
top_k = 10
top_logits, top_indices = torch.topk(final_logits, top_k)
print(f"\nTop {top_k} predicted tokens:")
for i, (logit_val, token_id) in enumerate(zip(top_logits, top_indices)):
    token_str = model.to_string(token_id)
    prob = probs[token_id].item()
    marker = "←" if token_id in [john_token, mary_token] else ""
    print(f"{i+1}. '{token_str}' - logit: {logit_val:.3f}, prob: {prob:.4f} {marker}")

Test prompt: When Mary and John went to the store, Mary gave

Logits shape: torch.Size([50257])
This is a score for each of the 50257 tokens in vocabulary

Token IDs:
' John' token ID: 1757
' Mary' token ID: 5335

Logit values:
Logit for ' John' (correct IO): 16.621
Logit for ' Mary' (subject, incorrect): 15.011
LOGIT DIFFERENCE: 1.609
Model prefers 'John' by 1.609 logits
  This means the model correctly favors the indirect object!

As probabilities:
P('John') = 0.1381 = 13.81%
P('Mary') = 0.0276 = 2.76%

Top 10 predicted tokens:
1. ' them' - logit: 17.976, prob: 0.5358 
2. ' John' - logit: 16.621, prob: 0.1381 ←
3. ' the' - logit: 15.432, prob: 0.0421 
4. ' a' - logit: 15.421, prob: 0.0416 
5. ' her' - logit: 15.141, prob: 0.0314 
6. ' Mary' - logit: 15.011, prob: 0.0276 ←
7. ' him' - logit: 14.496, prob: 0.0165 
8. ' us' - logit: 14.317, prob: 0.0138 
9. ' me' - logit: 13.915, prob: 0.0092 
10. ' birth' - logit: 13.758, prob: 0.0079 
